# Lesson 6 — Shipping to Production (Foundry + OPA sidecar)

Local notebooks are great for teaching. Production is HTTP, containers, 
and identity. This lesson packages the workshop agent as a container with 
an OPA sidecar and walks through the Foundry deployment manifest.

## Learning objectives
1. Understand the **sidecar** pattern (agent + OPA in one pod).
2. Recognize the HTTP endpoints Foundry will call.
3. Wire up OTEL tracing for production observability.
4. Walk a production rollout checklist.


In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
os.chdir(ROOT)               # so ./logs/policy-audit.jsonl lands in the standard location
sys.path.insert(0, str(ROOT))
from src.opa import OPAClient
opa = OPAClient()
assert opa.health_check(), 'OPA server unreachable on http://localhost:8181. Run: docker-compose up -d opa'
print('OPA reachable ✓   cwd =', pathlib.Path.cwd())

OPA reachable ✓   cwd = /home/anirudh/Projects/Rogers/agent-policy-opa


## 1. The sidecar pattern

```
          ┌──────────────────────────────────────────┐
          │            Foundry Agent Pod              │
          │                                          │
          │  ┌─────────────────┐   ┌──────────────┐  │
  HTTPS ──┼─▶│ agent_server.py │──▶│  OPA sidecar │  │
          │  │   :8000         │   │   :8181      │  │
          │  └─────────────────┘   └──────────────┘  │
          │           │                              │
          │           ▼                              │
          │   OTLP → Foundry tracing                 │
          └──────────────────────────────────────────┘
```
* OPA loads policies from a read-only volume mount → no agent restart on 
  policy change.
* The agent uses `DefaultAzureCredential` → Managed Identity in production, 
  `az login` locally.


## 2. The HTTP endpoints (from `src/server.py`)

| Method | Path     | Purpose                          |
|--------|----------|----------------------------------|
| GET    | /health  | Liveness probe                   |
| GET    | /tools   | List policy-enforced tools       |
| POST   | /tool    | Direct tool call (test / replay) |
| POST   | /invoke  | Natural-language entrypoint       |


## 3. Local smoke test

In [2]:
# Boot the server in another terminal:  uvicorn src.server:app --port 8000
# Then run the cell below.
import requests, contextlib
BASE = 'http://localhost:8000'
try:
    h = requests.get(f'{BASE}/health', timeout=2).json()
    print('health :', h)
    print('tools  :', requests.get(f'{BASE}/tools', timeout=2).json())
except requests.RequestException as e:
    print(f'(server not running — skip)  {e}')

(server not running — skip)  HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))


## 4. Wire tracing (OTLP → Foundry Toolkit viewer)

Set `ENABLE_TRACING=true` and `OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:4317` 
in `.env`. In production, swap the endpoint for the Foundry collector. 
Tracing setup is idempotent — see `src/observability.py::configure_tracing`.


In [3]:
from src.observability import configure_tracing
from src.config import get_settings
s = get_settings()
print('enable_tracing  :', s.enable_tracing)
print('otlp endpoint   :', s.otel_exporter_otlp_endpoint)
configure_tracing()  # safe to call multiple times

enable_tracing  : False
otlp endpoint   : http://localhost:4318


False

## 5. Deployment manifest essentials

See `.foundry/` in the repo for the full set. Required env vars:

| Variable | Source |
|----------|--------|
| `FOUNDRY_PROJECT_ENDPOINT` | Foundry-provided |
| `MODEL_API_VERSION` | `preview` (MAF needs `/openai/v1/`) |
| `MODEL_DEPLOYMENT_NAME` | e.g. `gpt-4o` |
| `OPA_URL` | `http://localhost:8181` (sidecar) |
| `ENABLE_POLICY_AUDIT` | `true` |
| `ENABLE_TRACING` | `true` |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Foundry collector |

Auth: bind a Managed Identity with the `Cognitive Services OpenAI User` role on 
the OpenAI resource. No API keys in production.


## 6. Production rollout checklist

- [ ] OPA sidecar mounted with `policies/*.rego` (read-only)
- [ ] Managed Identity bound, `disableLocalAuth=true` on OpenAI resource
- [ ] `ENABLE_POLICY_AUDIT=true`, audit volume mounted to persistent store
- [ ] OTEL exporter configured, traces visible in Foundry
- [ ] Health probe pointed at `/health`
- [ ] Rate limit (`OPA-B RATE_LIMIT`) tuned for expected traffic
- [ ] CI runs `pytest tests/` on every policy / agent change
- [ ] Runbook: how to add a role (edit Rego → restart OPA, no agent redeploy)


## Workshop recap

| Lesson | Case study | Key idea |
|--------|-----------|----------|
| 1 | Acme refund-bot, $48K loss | Unguarded tools are a liability |
| 2 | $2.4M PII fine | OPA-A: role + resource attributes |
| 3 | $1.8M price glitch | OPA-B: behaviour constraints |
| 4 | Refund-bot rebuilt | LLM + policy = safe tool use |
| 5 | 3 AM page | Audit log → reproduce → fix |
| 6 | Shipping | Sidecar, MI, tracing, checklist |

You now have a complete pattern for building AI agents whose tools are 
**provably** governed at runtime — without locking your governance team 
out of the change cycle.
